# Can Bitcoin be predicted in the short term?
### Prophet vs ARIMA vs LightGBM — evaluated against a naive baseline

Every model is compared with the simplest possible forecast — **"tomorrow's price = today's price"** — using
out-of-sample (walk-forward) testing. A model is only useful if it beats that baseline.

1. **Prophet** — trend + seasonality model on log price
2. **ARIMA** — classical autoregressive model, 3-day-ahead walk-forward test
3. **LightGBM** — next-day *direction* classifier with 18 technical / cross-asset features

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import logging; logging.getLogger("cmdstanpy").setLevel(logging.WARNING)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

START, END = "2015-01-01", "2026-10-02"   # fixed end date for reproducibility

## 0. Data

In [ ]:
raw = yf.download(["BTC-USD", "ETH-USD", "^GSPC"], start=START, end=END,
                  auto_adjust=True, progress=False)
close = raw["Close"].ffill()                       # fill weekend gaps of S&P 500
close = close[close["BTC-USD"].notna()]
volume = raw["Volume"]["BTC-USD"].reindex(close.index)
btc = close["BTC-USD"]

df = pd.DataFrame({"ds": btc.index, "y": np.log(btc.values)})   # Prophet format, log price
print(df.shape); close.tail(3)

## 1. Prophet

Improvements over the default model:
- **log transform** — BTC grows exponentially; without it the model predicts negative prices
- **`changepoint_range=0.95`** — default 0.8 ignores trend changes in the last 20% of the data
- **halving dates as holidays**, weekly seasonality off (crypto trades 24/7)

`changepoint_prior_scale=0.3, seasonality_prior_scale=1` were chosen with a grid search
(8 combinations × 31 CV folds).

In [ ]:
from prophet import Prophet
from prophet.diagnostics import cross_validation

halvings = pd.DataFrame({
    "holiday": "halving",
    "ds": pd.to_datetime(["2016-07-09", "2020-05-11", "2024-04-19"]),
    "lower_window": -60, "upper_window": 365,
})

def prophet_model(cps=0.3, sps=1, cp_range=0.95):
    return Prophet(changepoint_range=cp_range, changepoint_prior_scale=cps,
                   seasonality_prior_scale=sps, weekly_seasonality=False,
                   holidays=halvings)

m = prophet_model().fit(df)
forecast = m.predict(m.make_future_dataframe(60))
for c in ["yhat", "yhat_lower", "yhat_upper"]:
    forecast[c] = np.exp(forecast[c])

plt.figure(figsize=(12, 5))
plt.plot(df["ds"], np.exp(df["y"]), "k.", ms=2, label="Actual")
plt.plot(forecast["ds"], forecast["yhat"], label="Prophet")
plt.fill_between(forecast["ds"], forecast["yhat_lower"], forecast["yhat_upper"], alpha=0.3)
plt.yscale("log"); plt.legend(); plt.title("BTC-USD — Prophet (log scale)"); plt.show()

### 1a. 60-day cross-validation vs naive

In [ ]:
cv = cross_validation(m, initial="1460 days", period="90 days", horizon="60 days",
                      parallel="processes")
actual, pred = np.exp(cv["y"]), np.exp(cv["yhat"])
prophet_mape60 = (abs(actual - pred) / actual).mean() * 100

p = np.exp(df["y"])
naive_mape60 = (abs(p.shift(-60) - p) / p.shift(-60)).mean() * 100
print(f"Prophet 60-day MAPE: {prophet_mape60:.2f}%")
print(f"Naive   60-day MAPE: {naive_mape60:.2f}%")

### 1b. Short-term behaviour

Prophet fits a *smoothed* trend and does not use the latest price directly. In the last days of the data it sits
well below the actual price; making the trend more flexible fixes the fit but produces absurd extrapolations.

In [ ]:
recent = df.assign(actual=np.exp(df["y"]))[["ds", "actual"]]
rows = {}
for name, cps in [("rigid (0.3)", 0.3), ("flexible (0.5)", 0.5), ("medium (0.1)", 0.1)]:
    cp_range = 0.95 if cps == 0.3 else 0.99
    mm = prophet_model(cps=cps, cp_range=cp_range).fit(df)
    fc = mm.predict(mm.make_future_dataframe(60))
    fc["yhat"] = np.exp(fc["yhat"])
    last = fc.merge(recent, on="ds").iloc[-1]
    rows[name] = {
        "last actual": last["actual"],
        "fitted (last day)": last["yhat"],
        "bias %": (last["yhat"] / last["actual"] - 1) * 100,
        "60 days ahead": fc["yhat"].iloc[-1],
        "implied change %": (fc["yhat"].iloc[-1] / last["actual"] - 1) * 100,
    }
pd.DataFrame(rows).T.round(1)

## 2. ARIMA — 3-day-ahead walk-forward test

For each of the last 180 days: forecast 3 days ahead using only data up to that day, then add the new
observation (no refit) and move on.

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

y = df["y"].reset_index(drop=True)
TEST, H = 180, 3

res = ARIMA(y[:-TEST], order=(2, 1, 2)).fit()
err_arima, err_naive = [], []
for i in range(len(y) - TEST, len(y) - H + 1):
    actual = np.exp(y.iloc[i + H - 1])
    err_arima.append(abs(np.exp(res.forecast(H).iloc[-1]) - actual) / actual)
    err_naive.append(abs(np.exp(y.iloc[i - 1]) - actual) / actual)
    res = res.append(y.iloc[i:i + 1], refit=False)

arima_mape3 = np.mean(err_arima) * 100
naive_mape3 = np.mean(err_naive) * 100
print(f"ARIMA 3-day MAPE: {arima_mape3:.2f}%")
print(f"Naive 3-day MAPE: {naive_mape3:.2f}%")

In [ ]:
final_arima = ARIMA(y, order=(2, 1, 2)).fit()
f = final_arima.get_forecast(4)                    # data ends one day before END
ci = np.exp(f.conf_int(alpha=0.2))
pd.DataFrame({
    "ds": pd.date_range(df["ds"].iloc[-1] + pd.Timedelta(days=1), periods=4),
    "yhat": np.exp(f.predicted_mean).values,
    "yhat_lower": ci.iloc[:, 0].values,
    "yhat_upper": ci.iloc[:, 1].values,
}).round(0)

## 3. LightGBM — next-day direction

Target: does BTC close higher tomorrow? Features only use information available at the end of each day
(no look-ahead). Walk-forward over the last 2 years, retraining every 30 days.

In [ ]:
import lightgbm as lgb
from sklearn.metrics import accuracy_score, roc_auc_score

c = close.loc["2017-11-10":]                       # ETH data starts Nov 2017
b, v = c["BTC-USD"], volume.loc[c.index]
r = np.log(b).diff()

d = pd.DataFrame(index=c.index)
for k in [1, 2, 3, 7, 14, 30]:
    d[f"ret_{k}"] = np.log(b / b.shift(k))
d["vol_7"] = r.rolling(7).std()
d["vol_30"] = r.rolling(30).std()
d["vol_ratio"] = d["vol_7"] / d["vol_30"]
delta = b.diff()
gain = delta.clip(lower=0).rolling(14).mean()
loss = (-delta.clip(upper=0)).rolling(14).mean()
d["rsi"] = 100 - 100 / (1 + gain / loss)
for k in [20, 50, 200]:
    d[f"ma{k}"] = b / b.rolling(k).mean() - 1
d["volume"] = np.log(v / v.rolling(30).mean())
d["eth_1"] = np.log(c["ETH-USD"]).diff()
d["eth_7"] = np.log(c["ETH-USD"]).diff(7)
d["spx_1"] = np.log(c["^GSPC"]).diff()
d["spx_5"] = np.log(c["^GSPC"]).diff(5)
d["weekday"] = d.index.dayofweek
d["next_ret"] = r.shift(-1)

X_cols = [col for col in d.columns if col != "next_ret"]
data = d.replace([np.inf, -np.inf], np.nan).dropna().copy()
data["target"] = (data["next_ret"] > 0).astype(int)
X_last = d[X_cols].iloc[[-1]]
print(data.shape, "| share of up days:", round(data["target"].mean(), 3))

In [ ]:
def lgb_model():
    return lgb.LGBMClassifier(n_estimators=300, learning_rate=0.02, num_leaves=15,
                              min_child_samples=50, subsample=0.8, subsample_freq=1,
                              colsample_bytree=0.8, verbose=-1)

start = len(data) - 730
proba = pd.Series(index=data.index[start:], dtype=float)
for s in range(start, len(data), 30):
    train, test = data.iloc[:s], data.iloc[s:s + 30]
    model = lgb_model().fit(train[X_cols], train["target"])
    proba.loc[test.index] = model.predict_proba(test[X_cols])[:, 1]

test = data.loc[proba.index]
pred = (proba > 0.5).astype(int)
acc = accuracy_score(test["target"], pred)
base = max(test["target"].mean(), 1 - test["target"].mean())
auc = roc_auc_score(test["target"], proba)

trades = pred.diff().abs().fillna(0)
strategy = pred * test["next_ret"] - trades * 0.001          # 0.1% fee per trade
bh_ret = (np.exp(test["next_ret"].sum()) - 1) * 100
st_ret = (np.exp(strategy.sum()) - 1) * 100

print(f"Accuracy          : {acc*100:.1f}%")
print(f"Baseline (majority): {base*100:.1f}%")
print(f"AUC               : {auc:.3f}  (0.5 = coin flip)")
print(f"Buy & hold return : {bh_ret:.1f}%")
print(f"Strategy return   : {st_ret:.1f}%")

## 4. Summary

In [ ]:
pd.DataFrame([
    ["Prophet", "60-day MAPE", f"{prophet_mape60:.2f}%", f"naive {naive_mape60:.2f}%"],
    ["ARIMA(2,1,2)", "3-day MAPE", f"{arima_mape3:.2f}%", f"naive {naive_mape3:.2f}%"],
    ["LightGBM", "direction accuracy", f"{acc*100:.1f}%", f"majority {base*100:.1f}%"],
    ["LightGBM", "AUC", f"{auc:.3f}", "coin flip 0.500"],
    ["LightGBM strategy", "2-year return", f"{st_ret:.1f}%", f"buy & hold {bh_ret:.1f}%"],
], columns=["Model", "Metric", "Result", "Baseline"])

**Conclusion:** none of the models beats the naive baseline in a meaningful way. ARIMA converges to
"tomorrow = today", and LightGBM's direction predictions are no better than a coin flip. Using public price
data, short-term BTC price movements behave like a random walk — consistent with the efficient market
hypothesis.

**Next step:** forecast *volatility* instead of direction (GARCH / LightGBM regression on |returns|), which is
known to be much more predictable because volatility clusters.